# 04 — Enrichissement géographique : couverture GeoNames
Source : GeoNames.org (CC BY 4.0). Objectif : mesurer la part des transactions
dont le commerçant peut être localisé (latitude/longitude).

In [ ]:
import csv
import sys
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
GEO = ROOT / "data" / "raw" / "geonames"
TX = ROOT / "data" / "processed" / "silver" / "transactions.parquet"

print("Dossier      :", Path.cwd())
print("GEO existe   :", GEO.exists())
print("TX existe    :", TX.exists())

con = duckdb.connect()

In [ ]:
US_COLS = ["country", "postal_code", "place", "state_name", "state_code", "county_name",
           "county_code", "admin3_name", "admin3_code", "lat", "lon", "accuracy"]
us_zip = pd.read_csv(GEO / "us_zip" / "US.txt", sep="\t", header=None, names=US_COLS,
                     dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False)

CITY_COLS = ["geonameid", "name", "asciiname", "alternatenames", "lat", "lon",
             "feature_class", "feature_code", "country_code", "cc2", "admin1", "admin2",
             "admin3", "admin4", "population", "elevation", "dem", "timezone", "modified"]
cities = pd.read_csv(GEO / "cities15000.txt", sep="\t", header=None, names=CITY_COLS,
                     dtype=str, quoting=csv.QUOTE_NONE, keep_default_na=False)

lignes = [l.split("\t") for l in (GEO / "countryInfo.txt").read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
countries = pd.DataFrame([[l[0], l[4]] for l in lignes], columns=["iso2", "country_name"])

print("Codes postaux US :", len(us_zip))
print("Villes           :", len(cities))
print("Pays             :", len(countries))
display(us_zip[["postal_code", "place", "state_code", "lat", "lon"]].head(3))
display(cities[["name", "asciiname", "country_code", "lat", "lon", "population"]].head(3))
display(countries.head(3))

In [ ]:
con.register("us_zip", us_zip)
con.register("cities", cities)
con.register("countries", countries)

con.sql(f"""
    WITH z AS (SELECT DISTINCT postal_code FROM us_zip)
    SELECT COUNT(*)                                          AS tx_usa,
           COUNT(*) FILTER (WHERE t.merchant_zip IS NULL)    AS sans_zip,
           COUNT(z.postal_code)                              AS zip_trouve,
           ROUND(100.0 * COUNT(z.postal_code) / COUNT(*), 2) AS couverture_pct
    FROM '{TX}' t
    LEFT JOIN z ON t.merchant_zip = z.postal_code
    WHERE t.merchant_us_state IS NOT NULL
""").df()

In [ ]:
pays = con.sql(f"""
    SELECT t.merchant_country AS pays, c.iso2, COUNT(*) AS transactions
    FROM '{TX}' t
    LEFT JOIN countries c ON t.merchant_country = c.country_name
    WHERE t.merchant_country IS NOT NULL AND t.merchant_country <> 'United States'
    GROUP BY t.merchant_country, c.iso2
    ORDER BY transactions DESC
""").df()

print("Pays etrangers distincts :", len(pays))
print("Pays NON trouves         :", int(pays["iso2"].isna().sum()))
display(pays[pays["iso2"].isna()])

In [ ]:
con.sql(f"""
    WITH f AS (
        SELECT t.merchant_city, c.iso2, COUNT(*) AS n
        FROM '{TX}' t
        JOIN countries c ON t.merchant_country = c.country_name
        WHERE t.merchant_country <> 'United States'
        GROUP BY t.merchant_city, c.iso2
    ),
    v AS (
        SELECT DISTINCT country_code, name AS city FROM cities
        UNION
        SELECT DISTINCT country_code, asciiname FROM cities
    )
    SELECT SUM(n)                                                       AS tx_etranger,
           SUM(n) FILTER (WHERE v.city IS NOT NULL)                     AS ville_trouvee,
           ROUND(100.0 * SUM(n) FILTER (WHERE v.city IS NOT NULL) / SUM(n), 2) AS couverture_pct
    FROM f
    LEFT JOIN v ON f.iso2 = v.country_code AND f.merchant_city = v.city
""").df()

In [ ]:
ALIAS_PAYS = {
    "Netherlands": "NL", "The Bahamas": "BS", "Czech Republic": "CZ",
    "Vatican City": "VA", "Macedonia": "MK", "Cote d'Ivoire": "CI",
    "Myanmar (Burma)": "MM", "East Timor (Timor-Leste)": "TL", "Swaziland": "SZ",
}

print("Codes alias absents de GeoNames :",
      [c for c in ALIAS_PAYS.values() if c not in set(countries["iso2"])] or "aucun")

pays_iso = pd.concat([
    countries.rename(columns={"country_name": "pays"}),
    pd.DataFrame(list(ALIAS_PAYS.items()), columns=["pays", "iso2"]),
], ignore_index=True)
print("Noms en double dans la table :", int(pays_iso["pays"].duplicated().sum()))
con.register("pays_iso", pays_iso)

con.sql(f"""
    SELECT COUNT(*) AS tx_etranger,
           COUNT(p.iso2) AS pays_trouve,
           COUNT(*) - COUNT(p.iso2) AS pays_non_trouve
    FROM '{TX}' t
    LEFT JOIN pays_iso p ON t.merchant_country = p.pays
    WHERE t.merchant_country IS NOT NULL AND t.merchant_country <> 'United States'
""").df()

In [ ]:
villes_absentes = con.sql(f"""
    WITH v AS (
        SELECT DISTINCT country_code, name AS city FROM cities
        UNION
        SELECT DISTINCT country_code, asciiname FROM cities
    )
    SELECT t.merchant_city, t.merchant_country, p.iso2, COUNT(*) AS transactions
    FROM '{TX}' t
    JOIN pays_iso p ON t.merchant_country = p.pays
    LEFT JOIN v ON p.iso2 = v.country_code AND t.merchant_city = v.city
    WHERE t.merchant_country <> 'United States' AND v.city IS NULL
    GROUP BY ALL
    ORDER BY transactions DESC
    LIMIT 15
""").df()

zips_absents = con.sql(f"""
    WITH z AS (SELECT DISTINCT postal_code FROM us_zip)
    SELECT t.merchant_zip, t.merchant_city, t.merchant_us_state, COUNT(*) AS transactions
    FROM '{TX}' t
    LEFT JOIN z ON t.merchant_zip = z.postal_code
    WHERE t.merchant_us_state IS NOT NULL AND z.postal_code IS NULL
    GROUP BY ALL
    ORDER BY transactions DESC
    LIMIT 10
""").df()

print("Villes etrangeres introuvables (top 15) :")
display(villes_absentes)
print("Codes postaux US introuvables (top 10) :")
display(zips_absents)

In [ ]:
import importlib
import time

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml.data import geo
importlib.reload(geo)

TX_GEO = ROOT / "data" / "processed" / "silver" / "tx_geo.parquet"

t0 = time.perf_counter()
n = geo.build_tx_geo(duckdb.connect(), TX, GEO, TX_GEO)
print(f"{n} lignes ecrites en {time.perf_counter() - t0:.0f} s")
print(f"Taille : {TX_GEO.stat().st_size / 1e6:.1f} Mo")


In [ ]:
con.sql(f"""
    SELECT geo_precision,
           COUNT(*)                                          AS transactions,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 3)  AS part_pct
    FROM '{TX_GEO}'
    GROUP BY geo_precision
    ORDER BY transactions DESC
""").df()

In [ ]:
display(con.sql(f"""
    SELECT COUNT(*) FILTER (WHERE merchant_lat NOT BETWEEN -90 AND 90
                               OR merchant_lon NOT BETWEEN -180 AND 180)       AS coord_hors_limites,
           COUNT(*) FILTER (WHERE geo_precision IN ('zip', 'city', 'region')
                               AND merchant_lat IS NULL)                        AS precision_sans_coord,
           COUNT(*) FILTER (WHERE geo_precision IN ('online', 'unknown')
                               AND merchant_lat IS NOT NULL)                    AS coord_inattendues
    FROM '{TX_GEO}'
""").df())

print("Precision par pays (etranger, top 10) :")
con.sql(f"""
    SELECT t.merchant_country AS pays, g.geo_precision, COUNT(*) AS transactions
    FROM '{TX}' t JOIN '{TX_GEO}' g USING (tx_id)
    WHERE t.merchant_country IS NOT NULL AND t.merchant_country <> 'United States'
    GROUP BY ALL
    ORDER BY transactions DESC
    LIMIT 10
""").df()

## Conclusion
Enrichissement par `secumon/ml/data/geo.py` (5 tests) — 24 386 900 lignes en 3 s → `silver/tx_geo.parquet`.

| Précision | Transactions | Part |
|---|---|---|
| zip (code postal US) | 21 498 694 | 88,16 % |
| online (pas de lieu) | 2 720 821 | 11,16 % |
| city (ville + État / ville + pays, noms alternatifs GeoNames) | 165 158 | 0,68 % |
| region (centre de l'État / plus grande ville du pays) | 2 227 | 0,01 % |
| unknown | 0 | 0 % |

- 9 noms de pays IBM corrigés par code ISO 3166-1 (ex. Czech Republic → CZ).
- Fautes d'orthographe du dataset (Tapei, Sao Paolo...) non corrigées à la main : repli `region`.
- Contrôles : 0 coordonnée hors limites, 0 incohérence précision/coordonnées, nombre de lignes conservé.
- Source : GeoNames.org, licence CC BY 4.0.